In [1]:
pip install tslearn

  Obtaining dependency information for tslearn from https://files.pythonhosted.org/packages/91/0a/ced63ba8a2c64b84b635192b1b960c5ab4530eea022d116ef3a91f6b6d53/tslearn-0.6.2-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/369.8 kB ? eta -:--:--
   - -------------------------------------- 10.2/369.8 kB ? eta -:--:--
   ------ -------------------------------- 61.4/369.8 kB 656.4 kB/s eta 0:00:01
   --------------------- ------------------ 194.6/369.8 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 369.8/369.8 kB 2.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score
from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.metrics import cdist_dtw

In [27]:
df_normal = pd.read_csv('C:\\Users\\Gjirafa\\Alberti\\AlbZa\\DS Lab2\\Dataset\\SWaT_Dataset_Normal_v0.csv')
df_attack = pd.read_csv('C:\\Users\\Gjirafa\\Alberti\\AlbZa\\DS Lab2\\Dataset\\SWaT_Dataset_Attack_v0 - Copy.csv')

In [28]:
# Convert timestamps to datetime 
df_normal['Timestamp'] = pd.to_datetime(df_normal['Timestamp'])

C:\Users\Gjirafa\AppData\Local\Temp\ipykernel_6548\782962931.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_normal['Timestamp'] = pd.to_datetime(df_normal['Timestamp'])


In [39]:
df_normal.head()

,Timestamp,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,...,P501,P502,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Normal/Attack
0,2015-12-22 16:00:00,2.470294,261.5804,2,2,1,244.3284,8.19008,306.101,2.471278,...,1,1,10.02948,0.0,4.277749,0.000256,1,1,1,Normal
1,2015-12-22 16:00:01,2.457163,261.1879,2,2,1,244.3284,8.19008,306.101,2.468587,...,1,1,10.02948,0.0,4.277749,0.000256,1,1,1,Normal
2,2015-12-22 16:00:02,2.439548,260.9131,2,2,1,244.3284,8.19008,306.101,2.467305,...,1,1,10.02948,0.0,4.277749,0.000256,1,1,1,Normal
3,2015-12-22 16:00:03,2.428338,260.2850,2,2,1,244.3284,8.19008,306.101,2.466536,...,1,1,10.02948,0.0,4.277749,0.000256,1,1,1,Normal
4,2015-12-22 16:00:04,2.424815,259.8925,2,2,1,244.4245,8.19008,306.101,2.466536,...,1,1,10.02948,0.0,4.277749,0.000256,1,1,1,Normal


In [38]:
df_normal.dtypes

Timestamp        datetime64[ns]
FIT101                  float64
LIT101                  float64
MV101                     int64
P101                      int64
P102                      int64
AIT201                  float64
AIT202                  float64
AIT203                  float64
FIT201                  float64
MV201                     int64
P201                      int64
P202                      int64
P203                      int64
P204                      int64
P205                      int64
P206                      int64
DPIT301                 float64
FIT301                  float64
LIT301                  float64
MV301                     int64
MV302                     int64
MV303                     int64
MV304                     int64
P301                      int64
P302                      int64
AIT401                  float64
AIT402                  float64
FIT401                  float64
LIT401                  float64
P401                      int64
P402    

In [52]:
df_attack = pd.read_csv('C:\\Users\\Gjirafa\\Alberti\\AlbZa\\DS Lab2\\Dataset\\SWaT_Dataset_Attack_v0 - Copy.csv')

In [53]:
df_attack['Timestamp'] = pd.to_datetime(df_attack['Timestamp'], format='mixed')

In [54]:
df_attack['Timestamp'] = df_attack['Timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

In [37]:
df_attack.head()

,Timestamp,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,...,P501,P502,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Normal/Attack
0,2015-12-28 10:00:00,2.427057,522.8467,2,2,1,262.0161,8.396437,328.6337,2.445391,...,2,1,250.8652,1.649953,189.5988,0.000128,1,1,1,Normal
1,2015-12-28 10:00:01,2.446274,522.8860,2,2,1,262.0161,8.396437,328.6337,2.445391,...,2,1,250.8652,1.649953,189.6789,0.000128,1,1,1,Normal
2,2015-12-28 10:00:02,2.489191,522.8467,2,2,1,262.0161,8.394514,328.6337,2.442316,...,2,1,250.8812,1.649953,189.6789,0.000128,1,1,1,Normal
3,2015-12-28 10:00:03,2.534350,522.9645,2,2,1,262.0161,8.394514,328.6337,2.442316,...,2,1,250.8812,1.649953,189.6148,0.000128,1,1,1,Normal
4,2015-12-28 10:00:04,2.569260,523.4748,2,2,1,262.0161,8.394514,328.6337,2.443085,...,2,1,250.8812,1.649953,189.5027,0.000128,1,1,1,Normal


In [40]:
# Drop unnecessary columns
columns_to_drop = ['AIT201', 'P201', 'FIT601', 'P601', 'P602', 'P603', 'Normal/Attack']
df_normal.drop(columns_to_drop, axis=1, inplace=True)
df_attack.drop(columns_to_drop, axis=1, inplace=True)

In [41]:
# Print the shape of the DataFrame (number of rows and columns)
print(f'Shape of normal dataset: {df_normal.shape}')
print(f'Shape of attack dataset: {df_attack.shape}')

print()

# Print the column names of the DataFrame
print(f'Columns of normal dataset: {df_normal.columns}')
print(f'Columns of attack dataset: {df_attack.columns}')

Shape of normal dataset: (496800, 46)
Shape of attack dataset: (449919, 46)

Columns of normal dataset: Index(['Timestamp', 'FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT202',
       'AIT203', 'FIT201', 'MV201', 'P202', 'P203', 'P204', 'P205', 'P206',
       'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304',
       'P301', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P401', 'P402',
       'P403', 'P404', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504',
       'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'P502', 'PIT501',
       'PIT502', 'PIT503'],
      dtype='object')
Columns of attack dataset: Index(['Timestamp', 'FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT202',
       'AIT203', 'FIT201', 'MV201', 'P202', 'P203', 'P204', 'P205', 'P206',
       'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304',
       'P301', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P401', 'P402',
       'P403', 'P404', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT

In [42]:
df_normal = df_normal.set_index('Timestamp')
df_attack = df_attack.set_index('Timestamp')

In [43]:
# Standardize the data separately for normal and attack datasets
scaler_normal = StandardScaler()
scaler_attack = StandardScaler()

In [44]:
df_normal = scaler_normal.fit_transform(df_normal)
df_attack = scaler_attack.fit_transform(df_attack)

In [45]:
df_normal = pd.DataFrame(df_normal)
df_attack = pd.DataFrame(df_attack)

# Split the dataset into training and testing
train_data = df_normal  # Use the first 7 days for training
test_data = pd.concat([df_normal, df_attack])  # Combine both datasets for testing

In [46]:
# Create Time Series K-Means model with DTW metric
n_clusters = 2
model = TimeSeriesKMeans(n_clusters=n_clusters, metric='dtw', verbose=True)

# Train the model on normal operation data
model.fit(train_data)


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done 496800 out of 496800 | elapsed:  1.2min finished
[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done 993600 out of 993600 | elapsed:  3.8min finished
[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done 993600 out of 993600 | elapsed:  4.0min finished


20.811 --> 

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done 993600 out of 993600 | elapsed:  4.1min finished


18.560 --> 

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done 993600 out of 993600 | elapsed:  3.8min finished


...

18.466 --> 

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done 993600 out of 993600 | elapsed:  3.6min finished


18.466 --> 


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done 993600 out of 993600 | elapsed:  3.6min finished


TimeSeriesKMeans(metric='dtw', n_clusters=2, verbose=True)

In [49]:
pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [50]:
import joblib

model_filename = 'C:\\Users\\Gjirafa\\Alberti\\AlbZa\\DS Lab2\\Dataset\\anomaly_detectiontimeseries_kmeans_model.pkl'
joblib.dump(model, model_filename)

['C:\\Users\\Gjirafa\\Alberti\\AlbZa\\DS Lab2\\Dataset\\anomaly_detectiontimeseries_kmeans_model.pkl']

In [51]:
loaded_model = joblib.load(model_filename)


In [58]:
labels = loaded_model.predict(df_attack)

C:\Users\Gjirafa\anaconda3\Lib\site-packages\tslearn\utils\utils.py:90: UserWarning: 2-Dimensional data passed. Assuming these are 449919 1-dimensional timeseries
  warnings.warn(
[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done 899838 out of 899838 | elapsed:  2.9min finished


In [68]:
# Statistical Measures (Silhouette Score)
from sklearn.metrics import silhouette_score

silhouette_avg = silhouette_score(df_attack, labels)
print(f"Silhouette Score: {silhouette_avg}")

Silhouette Score: 0.4611441694688576
